In [ ]:
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import joblib

In [6]:
# Loading all crime data and clustering
df_crimes = pd.read_csv("../data/processed/all_data_uncleaned.csv").copy()
df_crimes = df_crimes.dropna(subset=["LSOA code"])
df_crimes = df_crimes.dropna(subset=["Crime type"])
df_crimes = df_crimes.dropna(subset=["Month"])
df_crimes = df_crimes.drop(columns=["Context"])

df_crimes["Month"] = pd.to_datetime(df_crimes["Month"])
# df_crimes["Month"] = df_crimes["Month"].dt.month

df_crimes = (df_crimes.groupby(["LSOA code", "Month", "Crime type"]).size().reset_index(name="crime_count"))

In [7]:
le_crime = LabelEncoder()
le_lsoa  = LabelEncoder()

df_crimes["crime_type_enc"] = le_crime.fit_transform(df_crimes["Crime type"].fillna("Unknown"))
df_crimes["lsoa_enc"]       = le_lsoa.fit_transform(df_crimes["LSOA code"].fillna("Unknown"))
df_crimes["month_num"]      = pd.to_datetime(df_crimes["Month"]).dt.month

train = df_crimes[df_crimes["Month"] < "2025-01-01"]
test  = df_crimes[df_crimes["Month"] >= "2025-01-01"]

features = ["crime_type_enc", "month_num", "lsoa_enc"]
target   = "crime_count"

X_train, y_train = train[features], train[target]
X_test,  y_test  = test[features],  test[target]

model = xgb.XGBRegressor(
    objective="count:poisson",
    n_estimators=800,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=10,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.5,
    gamma=0.1,
    random_state=42
)
model.fit(X_train, y_train)

preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f"RMSE: {rmse:.2f}")

RMSE: 5.26


In [8]:
joblib.dump(model, "../models/prediction_model.pkl")

['../models/prediction_model.pkl']

In [ ]:

# Generate predictions for every London LSOA x crime type x month (2025) and export to CSV
import itertools
import sys
sys.path.append("../../backend/services")
from crime_weight import crime_weights

lsoa_station = pd.read_csv("../data/shape/lsoa_station_map.csv", usecols=["LSOA21CD", "station"])
police_station = pd.read_csv("../data/shape/police_stations.csv", usecols=["station", "police_force"])

lsoa_station = (
    lsoa_station
    .merge(
        police_station,
        on="station",
        how="left"   # keeps all LSOA_CODE rows
    )
    [["LSOA21CD", "station", "police_force"]]
)

all_lsoas       = df_crimes["LSOA code"].unique()
all_crime_types = df_crimes["Crime type"].unique()
months          = list(range(1, 13))

rows = list(itertools.product(all_lsoas, all_crime_types, months))
pred_df = pd.DataFrame(rows, columns=["LSOA code", "Crime type", "month_num"])
pred_df["year"] = 2025

pred_df["lsoa_enc"]       = le_lsoa.transform(pred_df["LSOA code"])
pred_df["crime_type_enc"] = le_crime.transform(pred_df["Crime type"])

pred_df["predicted_crime_count"] = model.predict(pred_df[["crime_type_enc", "month_num", "lsoa_enc"]])
pred_df["predicted_crime_count"] = pred_df["predicted_crime_count"].clip(lower=0).round().astype(int)

pred_df["severity_score"] = pred_df["Crime type"].map(crime_weights)
pred_df["predicted_crime_severity"] = pred_df["predicted_crime_count"] * pred_df["severity_score"]

# Inner join — keeps only London LSOAs present in lsoa_station_map
pred_df = pred_df.merge(lsoa_station, left_on="LSOA code", right_on="LSOA21CD", how="inner").drop(columns="LSOA21CD")

out_df = pred_df[["LSOA code", "station", "police_force", "Crime type", "year", "month_num", "predicted_crime_count", "predicted_crime_severity"]].rename(columns={"month_num": "month"})
out_df = out_df[out_df["police_force"] == "metropolitan"]
out_path = "../data/processed/lsoa_crime_predictions.csv"
out_df.to_csv(out_path, index=False)
print(f"Saved {len(out_df):,} rows to {out_path}")
out_df.head(10)


Saved 2,641,296 rows to ../data/processed/lsoa_crime_predictions.csv


,LSOA code,station,police_force,Crime type,year,month,predicted_crime_count,predicted_crime_severity
0,E01000001,Islington Police Station,metropolitan,Burglary,2025,1,2,730.0
1,E01000001,Islington Police Station,metropolitan,Burglary,2025,2,2,730.0
2,E01000001,Islington Police Station,metropolitan,Burglary,2025,3,2,730.0
3,E01000001,Islington Police Station,metropolitan,Burglary,2025,4,2,730.0
4,E01000001,Islington Police Station,metropolitan,Burglary,2025,5,2,730.0
5,E01000001,Islington Police Station,metropolitan,Burglary,2025,6,2,730.0
6,E01000001,Islington Police Station,metropolitan,Burglary,2025,7,2,730.0
7,E01000001,Islington Police Station,metropolitan,Burglary,2025,8,2,730.0
8,E01000001,Islington Police Station,metropolitan,Burglary,2025,9,2,730.0
9,E01000001,Islington Police Station,metropolitan,Burglary,2025,10,2,730.0


In [ ]:

import sys
sys.path.append("../../backend/services")
from budget_allocation import allocate_budget

budget_df = allocate_budget(
    predictions_path="../data/processed/lsoa_crime_predictions.csv",
    output_path="../data/processed/budget_allocation.csv",
)
print(f"Saved {len(budget_df):,} rows to ../data/processed/budget_allocation.csv")
budget_df.head(20)
